# [Task 01] API를 활용한 로켓 발사 데이터 ETL 프로세스 구현

- **목표**: 외부 API에서 로켓 발사 데이터를 수집(Extract) → 변환(Transform) → 저장(Load)
- **사용 라이브러리**: requests, pandas, os, json, matplotlib, PIL

In [2]:
import requests
import json
import os
import pandas as pd

## 단계 1: 데이터 수집 및 JSON 저장 (load_data_from_api)

In [6]:
def load_data_from_api():
    """API에서 로켓 발사 데이터를 수집하고 JSON 파일로 저장한 뒤 DataFrame으로 반환"""
    
    url = "https://ll.thespacedevs.com/2.0.0/launch/upcoming"
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()
        print(f"API 호출 성공! 상태 코드: {response.status_code}")
        print(f"전체 발사 예정 수: {data['count']}")
        print(f"이번 페이지 데이터 수: {len(data['results'])}")
    except requests.exceptions.ConnectionError:
        print("네트워크 연결 오류가 발생했습니다.")
        return None
    except requests.exceptions.Timeout:
        print("요청 시간이 초과되었습니다.")
        return None
    except requests.exceptions.HTTPError as e:
        print(f"HTTP 에러 발생: {e}")
        return None
    
    # data 폴더 생성
    os.makedirs("data", exist_ok=True)
    
    # JSON 파일로 저장
    with open("data/launches.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print("data/launches.json 저장 완료")
    
    # DataFrame으로 변환하여 반환
    df = pd.DataFrame(data["results"])
    return df

In [7]:
df = load_data_from_api()

if df is not None:
    print(f"\nDataFrame 크기: {df.shape}")
    print(f"\n컬럼 목록:")
    print(df.columns.tolist())
    print()
    df.head()

HTTP 에러 발생: 429 Client Error: Too Many Requests for url: https://ll.thespacedevs.com/2.0.0/launch/upcoming/


In [8]:
if df is not None:
    df.info()

## 단계 2: 이미지 데이터 추출 및 다운로드 (get_pictures)

In [9]:
def get_pictures():
    """launches.json에서 이미지 URL을 추출하고 다운로드"""
    
    # JSON 파일 읽기
    with open("data/launches.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    
    launches = data["results"]
    print(f"총 {len(launches)}개의 발사 데이터에서 이미지 추출 시작")
    
    # 이미지 URL 추출 (없는 경우 건너뜀)
    image_urls = []
    for launch in launches:
        image_url = launch.get("image")
        if image_url:
            image_urls.append({
                "name": launch.get("name", "unknown"),
                "url": image_url
            })
        else:
            print(f"  이미지 없음: {launch.get('name', 'unknown')}")
    
    print(f"\n이미지가 있는 항목: {len(image_urls)}개")
    
    # images 폴더 생성
    os.makedirs("images", exist_ok=True)
    
    # 이미지 다운로드
    downloaded = 0
    for i, item in enumerate(image_urls):
        # 파일명 생성: 특수문자 제거 후 안전한 이름 만들기
        safe_name = item["name"].replace("/", "_").replace("\\", "_").replace(" ", "_")
        safe_name = safe_name[:50]  # 너무 긴 이름 방지
        filename = f"images/{safe_name}.jpg"
        
        try:
            img_response = requests.get(item["url"], timeout=15)
            img_response.raise_for_status()
            
            with open(filename, "wb") as f:
                f.write(img_response.content)
            
            downloaded += 1
            print(f"  [{downloaded}] 다운로드 완료: {safe_name}")
            
        except requests.exceptions.RequestException as e:
            print(f"  다운로드 실패: {safe_name} - {e}")
    
    print(f"\n총 {downloaded}개 이미지 다운로드 완료 (images/ 폴더)")
    return downloaded

In [10]:
downloaded_count = get_pictures()

FileNotFoundError: [Errno 2] No such file or directory: 'data/launches.json'

## 단계 3: 이미지 데이터 읽어와 시각화

In [11]:
import matplotlib.pyplot as plt
from PIL import Image
import math

In [12]:
def visualize_images():
    """images 폴더의 이미지를 그리드로 시각화"""
    
    # 이미지 파일 목록 가져오기
    image_files = [f for f in os.listdir("images") if f.endswith((".jpg", ".png", ".jpeg"))]
    image_files.sort()
    
    if not image_files:
        print("images 폴더에 이미지가 없습니다.")
        return
    
    print(f"총 {len(image_files)}개 이미지 시각화")
    
    # 그리드 크기 계산 (5열 기준)
    cols = 5
    rows = math.ceil(len(image_files) / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
    fig.suptitle("Upcoming Rocket Launches - Images", fontsize=16, fontweight="bold")
    
    # axes를 1차원 배열로 변환 (행이 1개일 때도 동일하게 처리)
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    axes_flat = [ax for row in (axes if rows > 1 else [axes]) for ax in (row if hasattr(row, '__iter__') else [row])]
    
    for i, ax in enumerate(axes_flat):
        if i < len(image_files):
            try:
                img = Image.open(os.path.join("images", image_files[i]))
                ax.imshow(img)
                # 파일명에서 .jpg 제거 후 제목 표시
                title = image_files[i].replace(".jpg", "").replace("_", " ")[:30]
                ax.set_title(title, fontsize=8)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error:\n{e}", ha="center", va="center", fontsize=8)
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_images()

---
### 결과 확인

In [ ]:
# data 폴더와 images 폴더에 파일이 정상적으로 존재하는지 확인
print("=== data 폴더 ===")
for f in os.listdir("data"):
    size = os.path.getsize(os.path.join("data", f))
    print(f"  {f} ({size:,} bytes)")

print(f"\n=== images 폴더 ({len(os.listdir('images'))}개) ===")
for f in sorted(os.listdir("images")):
    size = os.path.getsize(os.path.join("images", f))
    print(f"  {f} ({size:,} bytes)")